# NorgesGruppen Object Detection — Optimal trening på Google Colab

**Gjør dette FØR du starter:**
1. `Runtime → Change runtime type → T4 GPU` → Save
2. Sørg for at disse to filene ligger i **My Drive** (ikke i en mappe):
   - `NM_NGD_coco_dataset.zip`
   - `NM_NGD_product_images.zip`

**Kjør deretter cellene én etter én (▶)**

In [ ]:
# Steg 1: Sjekk GPU og installer pakker
!nvidia-smi
!pip install ultralytics -q
print('\nPakker installert!')

In [ ]:
# Steg 2: Koble til Google Drive og kopier begge zip-filer
from google.colab import auth
auth.authenticate_user()

from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy('/content/drive/MyDrive/NM_NGD_coco_dataset.zip',    'NM_NGD_coco_dataset.zip')
shutil.copy('/content/drive/MyDrive/NM_NGD_product_images.zip',  'NM_NGD_product_images.zip')
print('Begge zip-filer er klare!')

In [ ]:
# Steg 3: Pakk ut begge zip-filer
import zipfile

print('Pakker ut treningsdata...')
with zipfile.ZipFile('NM_NGD_coco_dataset.zip', 'r') as z:
    z.extractall('data/coco')

print('Pakker ut produktbilder...')
with zipfile.ZipFile('NM_NGD_product_images.zip', 'r') as z:
    z.extractall('data/product_images')

print('Ferdig pakket ut!')

In [ ]:
# Steg 4: Konverter COCO-format til YOLO-format (95% trening, 5% validering)
import json
import shutil
from pathlib import Path
from collections import defaultdict

# Finn annotations.json og bildemappen
ann_path = None
img_dir  = None
for root, dirs, files_list in __import__('os').walk('data/coco'):
    for f in files_list:
        if f == 'annotations.json':
            ann_path = Path(root) / f
            p = Path(root)
            if   (p / 'images').exists():        img_dir = p / 'images'
            elif (p.parent / 'images').exists(): img_dir = p.parent / 'images'
            break
    if ann_path:
        break

print(f'Annotations: {ann_path}')
print(f'Bilder:      {img_dir}')

with open(ann_path) as f:
    coco = json.load(f)

print(f"Bilder: {len(coco['images'])}, Kategorier: {len(coco['categories'])}, Annotasjoner: {len(coco['annotations'])}")

# Bygg oppslagstabeller
images_info   = {img['id']: img for img in coco['images']}
cat_id_to_idx = {cat['id']: i   for i, cat in enumerate(coco['categories'])}
names         = [cat['name']    for cat in coco['categories']]

img_anns = defaultdict(list)
for ann in coco['annotations']:
    if not ann.get('iscrowd', 0):
        img_anns[ann['image_id']].append(ann)

# Lag mapper
yolo_dir = Path('data/yolo')
for split in ('train', 'val'):
    (yolo_dir / 'images' / split).mkdir(parents=True, exist_ok=True)
    (yolo_dir / 'labels' / split).mkdir(parents=True, exist_ok=True)

# 95/5-split: mer treningsdata gir bedre modell
all_ids   = list(images_info.keys())
split_idx = int(len(all_ids) * 0.95)
train_ids = set(all_ids[:split_idx])
val_ids   = set(all_ids[split_idx:])

def write_split(ids, split_name):
    for img_id in ids:
        info = images_info[img_id]
        src  = img_dir / info['file_name']
        if not src.exists():
            continue
        W, H = info['width'], info['height']
        shutil.copy2(src, yolo_dir / 'images' / split_name / info['file_name'])
        lines = []
        for ann in img_anns.get(img_id, []):
            x, y, w, h = ann['bbox']
            cx = max(0.0, min(1.0, (x + w/2) / W))
            cy = max(0.0, min(1.0, (y + h/2) / H))
            nw = max(0.0, min(1.0, w / W))
            nh = max(0.0, min(1.0, h / H))
            lines.append(f"{cat_id_to_idx[ann['category_id']]} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
        (yolo_dir / 'labels' / split_name / (src.stem + '.txt')).write_text('\n'.join(lines))
    print(f'  {split_name}: {len(ids)} bilder')

write_split(train_ids, 'train')
write_split(val_ids,   'val')
print('Konvertering ferdig!')

In [ ]:
# Steg 5: Lag dataset.yaml
import yaml

cfg = {
    'path':  str(yolo_dir.resolve()),
    'train': 'images/train',
    'val':   'images/val',
    'nc':    len(names),
    'names': names,
}
yaml_path = yolo_dir / 'dataset.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(cfg, f, allow_unicode=True)
print(f'Lagret {yaml_path}')
print(f'Klasser: {len(names)}, Treningsbilder: {len(train_ids)}, Valideringsbilder: {len(val_ids)}')

In [ ]:
# Steg 6: TREN MODELLEN med optimale innstillinger
# YOLOv8l — stor, presis, passer T4 GPU
# Tar 2-4 timer. Ikke lukk nettleserfanen!

from ultralytics import YOLO
import torch

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

model = YOLO('yolov8l.pt')

model.train(
    data=str(yaml_path),
    epochs=200,
    imgsz=1280,
    batch=4,
    workers=2,
    project='runs',
    name='norgesgruppen',
    patience=30,           # stopper tidlig hvis ingen forbedring
    cos_lr=True,           # cosine learning rate
    warmup_epochs=3,
    label_smoothing=0.1,   # hjelper med mange klasser (356 stk)
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=0.0,
    translate=0.1,
    scale=0.5,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    close_mosaic=10,
    save=True,
    save_period=-1,
)

# Kopier beste vekter til rotnivå
import shutil
best_src = next(Path('runs').rglob('best.pt'))
shutil.copy2(best_src, 'best.pt')
print(f'\nTrening ferdig! Beste vekter: {best_src}')

In [ ]:
# Steg 7: Bygg reference embeddings (forbedrer klassifisering med produktbilder)
import json
import numpy as np
import torch
import torchvision.models as tvm
from PIL import Image
from pathlib import Path
from collections import defaultdict

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Bruker: {device}')

# Last inn MobileNetV3-small (uten klassifiseringshode)
feat_model = tvm.mobilenet_v3_small(weights=tvm.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
feat_model.classifier = torch.nn.Identity()
feat_model.eval().to(device)
torch.save(feat_model.state_dict(), 'feature_extractor.pt')
print('feature_extractor.pt lagret')

# Les annotations for å finne product_code → category_id
with open(ann_path) as f:
    coco2 = json.load(f)

cat_to_codes = defaultdict(set)
for ann in coco2['annotations']:
    code = ann.get('product_code')
    if code:
        cat_to_codes[ann['category_id']].add(str(code))
print(f'{len(cat_to_codes)} kategorier med produktkoder')

# Embed-funksjon
VIEWS    = ['main', 'front', 'back', 'left', 'right', 'top', 'bottom']
IMG_EXTS = ['.jpg', '.jpeg', '.png']
mean_t   = torch.tensor([0.485, 0.456, 0.406], device=device).view(1,3,1,1)
std_t    = torch.tensor([0.229, 0.224, 0.225], device=device).view(1,3,1,1)

def embed_image(img_path):
    try:
        img = Image.open(img_path).convert('RGB').resize((224, 224))
        t   = torch.from_numpy(np.array(img, dtype=np.float32) / 255.0)
        t   = t.permute(2,0,1).unsqueeze(0).to(device)
        t   = (t - mean_t) / std_t
        with torch.no_grad():
            return feat_model(t).squeeze(0).cpu()
    except:
        return None

# Bygg embeddings for alle kategorier
refs_dir   = Path('data/product_images')
embeddings = []
labels     = []

for cat_id, codes in sorted(cat_to_codes.items()):
    cat_embs = []
    for code in codes:
        product_dir = refs_dir / code
        if not product_dir.exists():
            continue
        for view in VIEWS:
            for ext in IMG_EXTS:
                p = product_dir / f'{view}{ext}'
                if p.exists():
                    emb = embed_image(p)
                    if emb is not None:
                        cat_embs.append(emb)
                    break
    if not cat_embs:
        continue
    avg = torch.stack(cat_embs).mean(dim=0)
    avg = avg / (avg.norm() + 1e-8)
    embeddings.append(avg.numpy())
    labels.append(cat_id)
    if len(labels) % 50 == 0:
        print(f'  {len(labels)} kategorier behandlet...')

emb_matrix = np.stack(embeddings).astype(np.float32)
np.save('reference_embeddings.npy', emb_matrix)
with open('reference_labels.json', 'w') as f:
    json.dump(labels, f)

print(f'\nFerdig! reference_embeddings.npy: {emb_matrix.shape}')
print(f'reference_labels.json: {len(labels)} kategorier')

In [ ]:
# Steg 8: Last ned alle 4 filer til din PC
from google.colab import files
from pathlib import Path

required = {
    'best.pt':                  'Trente modellvekter',
    'feature_extractor.pt':     'MobileNetV3 feature extractor',
    'reference_embeddings.npy': 'Produkt-embeddings',
    'reference_labels.json':    'Embedding-labels',
}

print('Sjekker at alle filer finnes...')
for filename, desc in required.items():
    p = Path(filename)
    if p.exists():
        size_mb = p.stat().st_size / 1e6
        print(f'  OK  {filename} ({size_mb:.1f} MB) — {desc}')
    else:
        print(f'  MANGLER: {filename} — kjør forrige celle på nytt!')

print('\nLaster ned...')
for filename in required:
    if Path(filename).exists():
        files.download(filename)

print('\nALLE FILER LASTET NED!')
print('Neste steg: Se instruksjonene nedenfor.')

## Etter nedlasting — gjør dette på din PC

Du har nå 4 filer lastet ned. Gjør følgende:

1. **Flytt alle 4 filer** til repo-mappen (`NorgesGruppen-Data-Object-Detection/`)

2. **Lag submission-zip** — åpne terminal i den mappen og kjør:
   ```bash
   bash zip_submission.sh
   ```
   Dette lager `submission.zip` med alle nødvendige filer automatisk.

3. **Last opp** `submission.zip` på [competition-siden](https://app.ainm.no/submit/norgesgruppen-data)

**Filer som skal være i submission.zip:**
- `run.py`
- `requirements.txt`
- `best.pt`
- `feature_extractor.pt`
- `reference_embeddings.npy`
- `reference_labels.json`